In [ ]:
import numpy as np
import pandas as pd

#loads data in
import gendata as gen

#To split data into stratified train and test sets
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

#for XGBoost model data format
from sklearn.preprocessing import LabelEncoder 

# Import Model
import xgboost as xgb

#for hyperparameter tuning
from sklearn.model_selection import GridSearchCV

# Import Metrics
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import root_mean_squared_error as rmse
from scipy.special import huber
from sklearn.metrics import make_scorer, mean_pinball_loss

Loading/Preprocessing Data

In [ ]:
#read in the cleaned data file

df = gen.load_training_data()

<class 'pandas.DataFrame'>
RangeIndex: 5605266 entries, 0 to 5605265
Data columns (total 46 columns):
 #   Column                Dtype  
---  ------                -----  
 0   RECORD_ID             int64  
 1   THCIC_ID              int64  
 2   TYPE_OF_ADMISSION     float64
 3   SOURCE_OF_ADMISSION   str    
 4   PUBLIC_HEALTH_REGION  float64
 5   PAT_STATUS            int64  
 6   SEX_CODE              str    
 7   RACE                  int64  
 8   ETHNICITY             int64  
 9   ADMIT_WEEKDAY         float64
 10  LENGTH_OF_STAY        float64
 11  PAT_AGE               int64  
 12  EMERGENCY_DEPT_FLAG   str    
 13  DIAG_CODES_OA         str    
 14  CODE_1                int64  
 15  CODE_2                int64  
 16  CODE_3                int64  
 17  CODE_4                int64  
 18  CODE_5                int64  
 19  CODE_6                int64  
 20  CODE_7                int64  
 21  CODE_8                int64  
 22  CODE_9                int64  
 23  CODE_10           

In [ ]:
cat_features = gen.cat_features
cat_features

['TYPE_OF_ADMISSION',
 'SEX_CODE',
 'RACE',
 'ETHNICITY',
 'ADMIT_WEEKDAY',
 'EMERGENCY_DEPT_FLAG',
 'CODE_1',
 'CODE_2',
 'CODE_3',
 'CODE_4',
 'CODE_5',
 'CODE_6',
 'CODE_7',
 'CODE_8',
 'CODE_9',
 'CODE_10',
 'CODE_11',
 'CODE_12',
 'CODE_13',
 'CODE_14',
 'CODE_15',
 'CODE_16',
 'CODE_17',
 'CODE_18',
 'CODE_19',
 'CODE_20',
 'CODE_21',
 'CODE_22',
 'PAT_RURAL',
 'PROVIDER_RURAL',
 'QUARTER']

In [ ]:
#Define categorical features, and put into a data type that works with XGBoost
df[cat_features] = (
    df[cat_features]
    .fillna("Missing")
    .astype('category')
)

In [ ]:
#Make list of all features to be used in the model, including categorical features and numerical features
features = cat_features.copy()
features.extend(['PAT_AGE','LENGTH_OF_STAY','PAT_LATITUDE','PAT_LONGITUDE','PROVIDER_LATITUDE','PROVIDER_LONGITUDE','NUM_CODES','PAT2PROV_DISTANCE'])

features

['TYPE_OF_ADMISSION',
 'SEX_CODE',
 'RACE',
 'ETHNICITY',
 'ADMIT_WEEKDAY',
 'EMERGENCY_DEPT_FLAG',
 'CODE_1',
 'CODE_2',
 'CODE_3',
 'CODE_4',
 'CODE_5',
 'CODE_6',
 'CODE_7',
 'CODE_8',
 'CODE_9',
 'CODE_10',
 'CODE_11',
 'CODE_12',
 'CODE_13',
 'CODE_14',
 'CODE_15',
 'CODE_16',
 'CODE_17',
 'CODE_18',
 'CODE_19',
 'CODE_20',
 'CODE_21',
 'CODE_22',
 'PAT_RURAL',
 'PROVIDER_RURAL',
 'QUARTER',
 'PAT_AGE',
 'LENGTH_OF_STAY',
 'PAT_LATITUDE',
 'PAT_LONGITUDE',
 'PROVIDER_LATITUDE',
 'PROVIDER_LONGITUDE',
 'NUM_CODES',
 'PAT2PROV_DISTANCE']

Hyperparameter Tuning

In [ ]:
#Split data into a small subset for hyperparameter tuning
X_hyp_train, X_hyp_test, y_hyp_train, y_hyp_test = train_test_split(df[features],df.LENGTH_OF_STAY,
                                                    shuffle = True,
                                                    random_state = 1990,
                                                    stratify = df['STRATA'],
                                                   test_size = .975)

In [ ]:
#Calculate percentage of data with a stay length less than or equal to 30 days
quantile = df.loc[df.LENGTH_OF_STAY.astype(int) <= 30].shape[0]/df.shape[0]
quantile

0.987531903035467

In [ ]:
#Hyperparameter Tuning with RMSE
#Create validation set from the training data for hyperparameter tuning; required for XGBoost early stopping
X_hyp_tr, X_hyp_val, y_hyp_tr, y_hyp_val = train_test_split(X_hyp_train, y_hyp_train, test_size=0.2, random_state=42)

#encode target variable for XGBoost
le = LabelEncoder()
y_hyp_tr_encoded = le.fit_transform(y_hyp_tr)
y_hyp_val_encoded = le.fit_transform(y_hyp_val)

#hyperparameters that we want to tune/scan over
param_grid = {
    "max_depth": [4, 5],
    "learning_rate": [0.03, 0.05, .075, .1],
    "min_child_weight": [1, 3, 5],
    "max_cat_threshold": [16, 32, 64],
}

    
    # Instantiate Model for RMSE
model = xgb.XGBRegressor(
        n_estimators=1000,
        objective="reg:squarederror",
        eval_metric = "rmse",
        random_state=42,
        early_stopping_rounds=400,
        gamma=0,
        enable_categorical=True
    )

#Run the model over the hyperparameter grid to find the best combination of hyperparameters
grid_search = GridSearchCV(
        estimator = model,
        param_grid = param_grid,
        scoring = "neg_root_mean_squared_error",
        cv = 3,  
        verbose = 2,
        n_jobs = -1
    )

grid_search.fit(X_hyp_train, y_hyp_train, eval_set=[(X_hyp_val, y_hyp_val_encoded)],verbose=500)



Fitting 3 folds for each of 72 candidates, totalling 216 fits
[0]	validation_0-rmse:7.42009
[0]	validation_0-rmse:7.43534
[0]	validation_0-rmse:7.43384
[0]	validation_0-rmse:7.42009
[0]	validation_0-rmse:7.43277
[0]	validation_0-rmse:7.43534
[0]	validation_0-rmse:7.43531
[0]	validation_0-rmse:7.41208
[0]	validation_0-rmse:7.42747
[0]	validation_0-rmse:7.42594
[0]	validation_0-rmse:7.43534
[0]	validation_0-rmse:7.41376
[413]	validation_0-rmse:12.00241
[CV] END learning_rate=0.03, max_cat_threshold=16, max_depth=4, min_child_weight=1; total time=   9.5s
[420]	validation_0-rmse:7.72925
[CV] END learning_rate=0.03, max_cat_threshold=16, max_depth=5, min_child_weight=1; total time=  10.1s
[428]	validation_0-rmse:7.64659
[CV] END learning_rate=0.03, max_cat_threshold=16, max_depth=4, min_child_weight=1; total time=  10.4s
[411]	validation_0-rmse:12.80826
[423]	validation_0-rmse:7.67548
[421]	validation_0-rmse:8.97407
[CV] END learning_rate=0.03, max_cat_threshold=16, max_depth=4, min_child_w

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...state=42, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.03, 0.05, ...], 'max_cat_threshold': [16, 32, ...], 'max_depth': [4, 5], 'min_child_weight': [1, 3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation ti

In [ ]:
#Hyperparameter Tuning with MAE
#Create validation set from the training data for hyperparameter tuning; required for XGBoost early stopping
X_hyp_tr, X_hyp_val, y_hyp_tr, y_hyp_val = train_test_split(X_hyp_train, y_hyp_train, test_size=0.2, random_state=42)

#encode target variable for XGBoost
le = LabelEncoder()
y_hyp_tr_encoded = le.fit_transform(y_hyp_tr)
y_hyp_val_encoded = le.fit_transform(y_hyp_val)

#hyperparameters that we want to tune/scan over
param_grid = {
    "max_depth": [4, 5, 6],
    "learning_rate": [0.03, 0.05, .075],
    "min_child_weight": [1, 3],
    "max_cat_threshold": [16, 32],
}

    
    # Instantiate Model for MAE
model = xgb.XGBRegressor(
        n_estimators=1000,
        objective="reg:absoluteerror",
        eval_metric = "mae",
        random_state=42,
        early_stopping_rounds=400,
        gamma=0,
        enable_categorical=True
    )

#Run the model over the hyperparameter grid to find the best combination of hyperparameters
grid_search = GridSearchCV(
        estimator = model,
        param_grid = param_grid,
        scoring = "neg_mean_absolute_error",
        cv = 3,  
        verbose = 2,
        n_jobs = -1
    )

grid_search.fit(X_hyp_train, y_hyp_train, eval_set=[(X_hyp_val, y_hyp_val_encoded)], verbose = 500)



Fitting 3 folds for each of 36 candidates, totalling 108 fits
[0]	validation_0-mae:3.38525
[0]	validation_0-mae:3.38525
[0]	validation_0-mae:3.38525
[0]	validation_0-mae:3.38525
[0]	validation_0-mae:3.38525
[0]	validation_0-mae:3.38525
[0]	validation_0-mae:3.38525
[0]	validation_0-mae:3.38525
[0]	validation_0-mae:3.38525
[0]	validation_0-mae:3.38525
[0]	validation_0-mae:3.38525
[0]	validation_0-mae:3.38525
[500]	validation_0-mae:1.46165
[500]	validation_0-mae:1.44743
[500]	validation_0-mae:1.44210
[500]	validation_0-mae:1.43580
[500]	validation_0-mae:1.51801
[500]	validation_0-mae:1.51801
[500]	validation_0-mae:1.44743
[500]	validation_0-mae:1.43580
[500]	validation_0-mae:1.44355
[500]	validation_0-mae:1.44210
[500]	validation_0-mae:1.46165
[500]	validation_0-mae:1.44355
[804]	validation_0-mae:1.44743
[CV] END learning_rate=0.03, max_cat_threshold=16, max_depth=4, min_child_weight=1; total time=  55.8s
[0]	validation_0-mae:3.38525
[804]	validation_0-mae:1.43581
[CV] END learning_rate=0

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...teerror', ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.03, 0.05, ...], 'max_cat_threshold': [16, 32], 'max_depth': [4, 5, ...], 'min_child_weight': [1, 3]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for ea

In [51]:
#Hyperparameter Tuning with Quantile
#Create validation set from the training data for hyperparameter tuning; required for XGBoost early stopping
X_hyp_tr, X_hyp_val, y_hyp_tr, y_hyp_val = train_test_split(X_hyp_train, y_hyp_train, test_size=0.2, random_state=42)

#encode target variable for XGBoost
le = LabelEncoder()
y_hyp_tr_encoded = le.fit_transform(y_hyp_tr)
y_hyp_val_encoded = le.fit_transform(y_hyp_val)

#hyperparameters that we want to tune/scan over
param_grid = {
    "max_depth": [4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05],
    "min_child_weight": [3, 5, 7],
    "max_cat_threshold": [16, 32],
}

    
    # Instantiate Model for quantile
model = xgb.XGBRegressor(
        n_estimators=1000,
        objective="reg:quantileerror",
        quantile_alpha=quantile,
        #eval_metric = "",
        random_state=42,
        early_stopping_rounds=400,
        gamma=0,
        enable_categorical=True
    )

#Run the model over the hyperparameter grid to find the best combination of hyperparameters
grid_search = GridSearchCV(
        estimator = model,
        param_grid = param_grid,
        scoring=make_scorer(
            mean_pinball_loss,
            alpha=0.987531903035467,
            greater_is_better=False
            ),        
        cv = 3,  
        verbose = 2,
        n_jobs = -1
    )

grid_search.fit(X_hyp_train, y_hyp_train, eval_set=[(X_hyp_val, y_hyp_val_encoded)], verbose = 500)



Fitting 3 folds for each of 54 candidates, totalling 162 fits
[0]	validation_0-quantile:0.58863
[0]	validation_0-quantile:0.58488
[0]	validation_0-quantile:0.58984
[0]	validation_0-quantile:0.58863
[0]	validation_0-quantile:0.58984
[0]	validation_0-quantile:0.58984
[0]	validation_0-quantile:0.58488
[0]	validation_0-quantile:0.58863
[0]	validation_0-quantile:0.58984
[0]	validation_0-quantile:0.58488
[0]	validation_0-quantile:0.58488
[0]	validation_0-quantile:0.58863
[500]	validation_0-quantile:0.14037
[500]	validation_0-quantile:0.14254
[500]	validation_0-quantile:0.14336
[500]	validation_0-quantile:0.14137
[500]	validation_0-quantile:0.14061
[500]	validation_0-quantile:0.14136
[500]	validation_0-quantile:0.14185
[500]	validation_0-quantile:0.14000
[500]	validation_0-quantile:0.14002
[500]	validation_0-quantile:0.14230
[500]	validation_0-quantile:0.14000
[500]	validation_0-quantile:0.14039
[999]	validation_0-quantile:0.10308
[CV] END learning_rate=0.01, max_cat_threshold=16, max_depth=4

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...leerror', ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.01, 0.03, ...], 'max_cat_threshold': [16, 32], 'max_depth': [4, 5, ...], 'min_child_weight': [3, 5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",make_scorer(m...7531903035467)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation t

In [52]:
#Hyperparameter Tuning with Huber
#Create validation set from the training data for hyperparameter tuning; required for XGBoost early stopping
X_hyp_tr, X_hyp_val, y_hyp_tr, y_hyp_val = train_test_split(X_hyp_train, y_hyp_train, test_size=0.2, random_state=42)

#encode target variable for XGBoost
le = LabelEncoder()
y_hyp_tr_encoded = le.fit_transform(y_hyp_tr)
y_hyp_val_encoded = le.fit_transform(y_hyp_val)

#hyperparameters that we want to tune/scan over
param_grid = {
    "max_depth": [5,6, 7],
    "learning_rate": [0.1, 0.03, 0.05],
    "min_child_weight": [1, 3],
    "max_cat_threshold": [16, 32, 64],
}

#Function to define huber loss for use in GridSearchCV scoring
def huber_loss(y_true, y_pred, delta=2.0):
    error = y_true.astype(int) - y_pred.astype(int)
    is_small = np.abs(error) <= delta

    squared = 0.5 * error**2
    linear = delta * (np.abs(error) - 0.5 * delta)

    return np.mean(np.where(is_small, squared, linear))

huber_scorer = make_scorer(
    huber_loss,
    delta=2.0,
    greater_is_better=False)

# Instantiate Model for Huber
model = xgb.XGBRegressor(
        n_estimators=1000,
        objective="reg:pseudohubererror",
        eval_metric = "mphe",
        huber_slope = 2,
        random_state=42,
        early_stopping_rounds=400,
        gamma=0,
        enable_categorical=True
    )

#Run the model over the hyperparameter grid to find the best combination of hyperparameters
grid_search = GridSearchCV(
        estimator = model,
        param_grid = param_grid,
        scoring = huber_scorer,
        cv = 3,  
        verbose = 2,
        n_jobs = -1
    )

grid_search.fit(X_hyp_train, y_hyp_train, eval_set=[(X_hyp_val, y_hyp_val_encoded)], verbose = 500)



Fitting 3 folds for each of 54 candidates, totalling 162 fits
[0]	validation_0-mphe:11.01616
[0]	validation_0-mphe:11.18975
[0]	validation_0-mphe:11.10667
[0]	validation_0-mphe:10.25468
[0]	validation_0-mphe:10.96940
[0]	validation_0-mphe:10.21861
[0]	validation_0-mphe:11.11134
[0]	validation_0-mphe:10.26537
[0]	validation_0-mphe:10.17889
[0]	validation_0-mphe:10.10048
[0]	validation_0-mphe:11.05761
[0]	validation_0-mphe:10.20562
[500]	validation_0-mphe:0.64791
[500]	validation_0-mphe:0.70536
[500]	validation_0-mphe:1.81913
[500]	validation_0-mphe:1.12907
[500]	validation_0-mphe:0.80101
[500]	validation_0-mphe:0.66488
[500]	validation_0-mphe:1.66272
[500]	validation_0-mphe:1.40084
[500]	validation_0-mphe:1.25588
[500]	validation_0-mphe:0.99162
[500]	validation_0-mphe:0.92509
[500]	validation_0-mphe:0.60748
[999]	validation_0-mphe:0.67170
[999]	validation_0-mphe:1.63579
[CV] END learning_rate=0.1, max_cat_threshold=16, max_depth=5, min_child_weight=1; total time=  49.0s
[999]	validation

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.1, 0.03, ...], 'max_cat_threshold': [16, 32, ...], 'max_depth': [5, 6, ...], 'min_child_weight': [1, 3]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","make_scorer(h...t', delta=2.0)"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation 

Running Model (with tuned hyperparameters)

In [35]:
X_train, X_test, y_train, y_test = train_test_split(df[features],df.LENGTH_OF_STAY, shuffle = True, random_state = 2222, stratify = df['STRATA'], test_size = .2)

In [37]:
strata_train = df.loc[X_train.index, 'STRATA']

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2222
)

In [47]:
# Run the models for each metric on the train set, recording each metric.
# Dictionary defining different metrics and tuned hyperparameters [max_depth, learning_rate, min_child_weight, max_cat_threshold]
metrics = {"rmse" : [4,0.05,1,16],
           "mae": [5,0.05,1,16],
           "mphe": [7,0.03,1,16],
           "quantile": [5,0.01,5,16]}

scores = pd.DataFrame(
    0.0,
    index=["RMSE", "MAE", "Huber", "Quantile"],
    columns=["RMSE", "MAE", "Huber", "Quantile"]
)

row_names = {
    "rmse": "RMSE",
    "mae": "MAE",
    "mphe": "Huber",
    "quantile": "Quantile"
}

for metric,params in metrics.items():
    fold_rmse = []
    fold_mae = []
    fold_huber = []
    fold_quantile = []
    # Fit the model for each fold
    
    print(f"\nTraining {metric} model...")


    
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, strata_train)):
        X_tr = X_train.iloc[train_idx].copy()
        X_val = X_train.iloc[valid_idx].copy()
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[valid_idx]

        le = LabelEncoder()
        y_tr_encoded = le.fit_transform(y_tr)
        y_val_encoded = le.fit_transform(y_val)


        # Instantiate Model
        model = xgb.XGBRegressor(
            max_depth=params[0], #Tuned Hyperparameter 1
            learning_rate=params[1], #Tuned Hyperparameter 2
            min_child_weight=params[2], #Tuned Hyperparameter 3
            max_cat_threshold=params[3], #Tuned Hyperparameter 4
            n_estimators=4000,
            eval_metric=metric,
            random_state=42,
            early_stopping_rounds=400,
            enable_categorical=True,
            objective = "reg:squarederror" if metric == "rmse" else (
                "reg:absoluteerror" if metric == "mae" else (
                    "reg:pseudohubererror" if metric == "mphe" else (
                        "reg:quantileerror"
                    )
                )   
            ),
            quantile_alpha=quantile if metric == "quantile" else None,
            huber_slope=2 if metric == "mphe" else None,
        )

        model.fit(X_tr, y_tr_encoded, eval_set=[(X_val, y_val_encoded)], verbose=100)

        # Find predictions and store rmses  
        preds = model.predict(X_val)
    
        rmse_pred = rmse(y_val_encoded,preds)
        mae_pred = mae(y_val_encoded,preds)
        quantile_pred = mean_pinball_loss(
            y_val_encoded,
            preds,
            alpha=0.987531903035467
        )

        huber_pred = huber_loss(y_val_encoded, preds, delta=2.0)

        fold_rmse.append(rmse_pred)
        fold_mae.append(mae_pred)
        fold_huber.append(huber_pred)
        fold_quantile.append(quantile_pred)

        print(
        f"{metric} | Fold {fold+1}/{skf.get_n_splits()} | "
        f"RMSE={rmse_pred:.3f}, "
        f"MAE={mae_pred:.3f}, "
        f"Huber={huber_pred:.3f}, "
        f"Quantile={quantile_pred:.5f}"
        )

    row = row_names[metric]
    
    scores.loc[row, "RMSE"] = np.mean(fold_rmse)
    scores.loc[row, "MAE"] = np.mean(fold_mae)
    scores.loc[row, "Huber"] = np.mean(fold_huber)
    scores.loc[row, "Quantile"] = np.mean(fold_quantile)

    print(f"Finished {metric} model.\n")


Training rmse model...
[0]	validation_0-rmse:8.49816
[100]	validation_0-rmse:3.21067
[200]	validation_0-rmse:3.21038
[300]	validation_0-rmse:3.19876
[400]	validation_0-rmse:3.18915
[500]	validation_0-rmse:3.19084
[600]	validation_0-rmse:3.19385
[700]	validation_0-rmse:3.19966
[800]	validation_0-rmse:3.20266
[801]	validation_0-rmse:3.20271
rmse | Fold 1/5 | RMSE=3.189, MAE=0.164, Huber=0.525, Quantile=0.07425
[0]	validation_0-rmse:8.36329
[100]	validation_0-rmse:3.09866
[200]	validation_0-rmse:3.11360
[300]	validation_0-rmse:3.10252
[400]	validation_0-rmse:3.09528
[462]	validation_0-rmse:3.09635
rmse | Fold 2/5 | RMSE=3.078, MAE=0.281, Huber=0.436, Quantile=0.12788
[0]	validation_0-rmse:8.56631
[100]	validation_0-rmse:3.18398
[200]	validation_0-rmse:3.18617
[300]	validation_0-rmse:3.16442
[400]	validation_0-rmse:3.16649
[500]	validation_0-rmse:3.17102
[600]	validation_0-rmse:3.17670
[700]	validation_0-rmse:3.18838
[717]	validation_0-rmse:3.19038
rmse | Fold 3/5 | RMSE=3.163, MAE=0.164,

In [48]:
scores

,RMSE,MAE,Huber,Quantile
RMSE,3.115324,0.185299,0.498262,0.084030
MAE,5.117928,0.427539,0.648945,0.248210
Huber,7.027278,0.512911,1.177548,0.276349
Quantile,15.299836,7.735081,13.635617,0.097460
